# ComputerVisionAIHub · Train a YOLO26 model and publish it

This notebook takes you end-to-end:

1. Pull a labelled dataset from **Roboflow Universe**
2. Train a **YOLO26** model
3. Export both `.pt` and `.onnx` 
4. Push the weights to the **Hugging Face Hub**
5. Auto-print the `models.json` entry to paste into your catalog

> **Before you start:** Runtime → Change runtime type → **T4 GPU** (free).

## 0. Install dependencies

In [ ]:
!pip install -q ultralytics roboflow huggingface_hub

## 1. Get the dataset from Roboflow

Find a dataset on https://universe.roboflow.com, click **Download**, choose format
**YOLOv11** (the current Ultralytics export, fully compatible with YOLO26), and copy the snippet.
Paste your details below. Keep your API key private — don't commit it.

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="YOUR_ROBOFLOW_KEY")
project = rf.workspace("WORKSPACE").project("PROJECT")
dataset = project.version(1).download("yolov11")

DATA_YAML = f"{dataset.location}/data.yaml"
print("data.yaml ->", DATA_YAML)

## 2. Train

`yolo26n` (nano) is small and fast — great default. Bump to `yolo26s/m/l` for more accuracy
at the cost of size and speed. `epochs` and `imgsz` are main hyperparameters, for more look into Ultralytics YOLO docs.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26n.pt")
results = model.train(
    data=DATA_YAML,
    epochs=100,
    imgsz=640,
    project="runs",
    name="train",
)

## 3. Evaluate + export

Validation metrics (mAP) and export an ONNX copy alongside the `.pt`.

In [ ]:
# Validation metrics from the best checkpoint
metrics = model.val()
mAP50    = float(metrics.box.map50)
mAP50_95 = float(metrics.box.map)
print(f"mAP@50 = {mAP50:.4f} | mAP@50-95 = {mAP50_95:.4f}")

# Export a portable ONNX version (no Ultralytics needed to run it)
onnx_path = model.export(format="onnx")
print("ONNX ->", onnx_path)

import os, glob
BEST_PT   = "runs/train/weights/best.pt"
BEST_ONNX = "runs/train/weights/best.onnx"
print("pt size (MB):", round(os.path.getsize(BEST_PT)/1e6, 1))

## 4. Publish to the Hugging Face Hub

Create a **Write** token at huggingface.co → Settings → Access Tokens.
We create a public model repo and upload both files.

In [ ]:
from huggingface_hub import login, HfApi, ModelCard

HF_USER  = "lukasiktar"          # <- HF username
MODEL_ID = "traffic-signs-v1"  # <- unique id for this model
REPO     = f"{HF_USER}/{MODEL_ID}"

login(token="hf_XXXX")  # paste WRITE token

api = HfApi()
api.create_repo(REPO, repo_type="model", exist_ok=True)
api.upload_file(path_or_fileobj=BEST_PT,   path_in_repo="best.pt",   repo_id=REPO)
api.upload_file(path_or_fileobj=BEST_ONNX, path_in_repo="best.onnx", repo_id=REPO)

# A minimal model card so the HF page isn't empty
card = ModelCard(f"""---
tags: [object-detection, yolo, ultralytics]
license: agpl-3.0
---

# {MODEL_ID}

YOLO26 model trained for object detection. Part of the ComputerVisionAIHub catalog.
""")
card.push_to_hub(REPO)
print("Published ->", f"https://huggingface.co/{REPO}")

## 5. Get your `models.json` entry

Fill in the human-readable fields, run the cell, and paste the printed object into
`docs/models.json` (inside the `models` array). That's the only edit the website needs.

In [ ]:
import json
from ultralytics import YOLO as _Y

# class names come straight from the trained model
names = list(model.names.values())

entry = {
    "id": MODEL_ID,
    "name": "Traffic Sign Detector",            # <- edit
    "summary": "Detects common road signs.",     # <- edit
    "base_model": "yolo26n",
    "task": "detection",
    "classes": names,
    "dataset": "https://universe.roboflow.com/...",  # <- edit
    "dataset_license": "CC BY 4.0",                 # <- edit
    "metrics": {"mAP50": round(mAP50, 3), "mAP50_95": round(mAP50_95, 3)},
    "hf_repo": REPO,
    "download": f"https://huggingface.co/{REPO}/resolve/main/best.pt",
    "onnx": f"https://huggingface.co/{REPO}/resolve/main/best.onnx",
    "size_mb": round(os.path.getsize(BEST_PT)/1e6, 1),
    "image_size": 640,
}
print(json.dumps(entry, indent=2))